In [1]:
# For tips on running notebooks in Google Colab, see
# https://docs.pytorch.org/tutorials/beginner/colab
%matplotlib inline

In [2]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

torch_dtype=torch.float16

In [3]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


## Depedencies and requeriments (This chunk takes 5 min aprox)

In [4]:
# 1. AHN Project
!git clone https://github.com/ByteDance-Seed/AHN.git
!cd AHN

# 2. Install required forked libraries
!pip install "git+https://github.com/Seerkfang/flash-linear-attention.git@main#egg=flash-linear-attention"
!pip install "git+https://github.com/Seerkfang/LLaMA-Factory.git@main#egg=llamafactory"
!pip install -U "flash-attn==2.8.3" \
  --extra-index-url https://wheels.astral.sh/simple/cu128/
# (Optional) Install the forked Mamba version if you plan to use AHN-Mamba2
# MAMBA_FORCE_BUILD=TRUE pip install "git+https://github.com/yuweihao/mamba.git"

# 3. Install AHN in editable mode with training extras
!pip install -e ".[train]"

Cloning into 'AHN'...
remote: Enumerating objects: 80, done.
remote: Counting objects: 100% (80/80), done.
remote: Compressing objects: 100% (65/65), done.
remote: Total 80 (delta 18), reused 71 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (80/80), 91.63 KiB | 762.00 KiB/s, done.
Resolving deltas: 100% (18/18), done.
  Cloning https://github.com/Seerkfang/flash-linear-attention.git (to revision main) to /tmp/pip-install-3uuxrokq/flash-linear-attention_37cf0c04fd074e558467e8aa7f93d278
  Running command git clone --filter=blob:none --quiet https://github.com/Seerkfang/flash-linear-attention.git /tmp/pip-install-3uuxrokq/flash-linear-attention_37cf0c04fd074e558467e8aa7f93d278
  Resolved https://github.com/Seerkfang/flash-linear-attention.git to commit 95537c011c541185d6d1a5f78e1a60033695cc74
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached ninja-1.13.0-py3-none-manylinux2014_

# Set up base path project

In [1]:
%cd /content/AHN

/content/AHN


In [2]:
!pip install -e .

Obtaining file:///content/AHN
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for ahn (pyproject.toml) ... done
  Created wheel for ahn: filename=ahn-0.1.0-0.editable-py3-none-any.whl size=8691 sha256=6586ad8b219c10662a144534c7fcf9a7f0881dda3aeeb9311c65648b3181ecf8
  Stored in directory: /tmp/pip-ephem-wheel-cache-96l5swm9/wheels/93/e8/15/05cded1c488953ea1b130fc789276c1783947c96aef208e527
Successfully built ahn


## After install the next section, click in restart session button and continue run the code

In [3]:
!pip install --no-cache-dir --force-reinstall "transformers==4.51.0"
!pip install -U "datasets>=3.3.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 197.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 210.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.9/45.9 kB 292.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 175.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 380.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 259.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.0/130.0 kB 342.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.6/801.6 kB 189.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.8/801.8 kB 414.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 437.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 227.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 343.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.9/203.9 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 19.2 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2026.7.0
    Uninstalling fsspec-2026.7.0:
      Successfully uninstalled fsspec-2026.7.0
  Attempting uninstall: datasets
    Found existing installation: datasets 3.2.0
    Uninstalling datasets-3.2.0:
      Successfully uninstalled datasets-3.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
trl 0.9.6 requires numpy<2.0.0,>=1.18.2, but you have numpy 2.5.2 which is incompatible.
llamafactory 0.

------------------------------------------------------------------------


In [7]:
# Base model (repo_id or local path)
BASE_MODEL="Qwen/Qwen2.5-3B-Instruct"

# AHN-only weights (repo_id or local path)
AHN_PATH="ByteDance-Seed/AHN-GDN-for-Qwen-2.5-Instruct-3B"

# Output directory for the merged model
MERGED_MODEL_PATH="./merged_ckpt/Qwen-2.5-Instruct-3B-AHN-GDN"


!python /content/AHN/examples/scripts/utils/merge_weights.py \
    --base-model "{BASE_MODEL}" \
    --ahn-path "{AHN_PATH}" \
    --output-path "{MERGED_MODEL_PATH}"

2026-08-23 17:37:30.991995: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.











Some weights of Qwen2ForCausalLM were not initialized from the model checkpoint at Qwen/Qwen2.5-3B-Instruct and are newly initialized: ['model.layers.0.ahn.fn.A_log', 'model.layers.0.ahn.fn.D', 'model.layers.0.ahn.fn.a_proj.weight', 'model.layers.0.ahn.fn.b_proj.weight', 'model.layers.0.ahn.fn.dt_bias', 'model.layers.0.ahn.fn.g_proj.weight', 'model.layers.0.ahn.fn.o_norm.weight', 'model.layers.0.ahn.fn.o_proj.weight', 'model.layers.1.ahn.fn.A_log', 'model.layers.1.ahn.fn.D', 'model.layers.1.ahn.fn.a_proj.weight', 'model.layers.1.ahn.fn.b_proj.weight', 'model.layers.1.ahn.fn.dt_bias', 'model.layers.1.ahn.fn.g_proj.weight', 'model.layers.1.ahn.fn.o_norm.weight', 

In [8]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    MERGED_MODEL_PATH,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some weights of the model checkpoint at ./merged_ckpt/Qwen-2.5-Instruct-3B-AHN-GDN were not used when initializing Qwen2ForCausalLM: ['model.layers.0.ahn.fn.A_log', 'model.layers.0.ahn.fn.D', 'model.layers.0.ahn.fn.a_proj.weight', 'model.layers.0.ahn.fn.b_proj.weight', 'model.layers.0.ahn.fn.dt_bias', 'model.layers.0.ahn.fn.g_proj.weight', 'model.layers.0.ahn.fn.o_norm.weight', 'model.layers.0.ahn.fn.o_proj.weight', 'model.layers.1.ahn.fn.A_log', 'model.layers.1.ahn.fn.D', 'model.layers.1.ahn.fn.a_proj.weight', 'model.layers.1.ahn.fn.b_proj.weight', 'model.layers.1.ahn.fn.dt_bias', 'model.layers.1.ahn.fn.g_proj.weight', 'model.layers.1.ahn.fn.o_norm.weight', 'model.layers.1.ahn.fn.o_proj.weight', 'model.layers.10.ahn.fn.A_log', 'model.layers.10.ahn.fn.D', 'model.layers.10.ahn.fn.a_proj.weight', 'model.layers.10.ahn.fn.b_proj.weight', 'model.layers.10.ahn.fn.dt_bias', 'model.layers.10.ahn.fn.g_proj.weight', 'model.layers.10.ahn.fn.o_norm.weight', 'model.layers.10.ahn.fn.o_proj.weight', 

In [10]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    MERGED_MODEL_PATH,
    trust_remote_code=True,
)

In [11]:

from pathlib import Path
import torch
import torch.nn.functional as F
import random
import numpy as np
import pandas as pd


device = model.device

import re
def _normalize(s):
    """Lowercase, strip, remove surrounding punctuation/quotes/backticks and trailing junk."""
    s = s.strip()
    # cut at first newline or code-fence — model sometimes appends explanation/junk
    s = s.split("\n")[0]
    s = s.replace("`", " ")
    s = s.strip().strip('.').strip('"').strip("'").strip()
    return s.lower().strip()

def answer_matches(pred, truth):
    p, t = _normalize(pred), _normalize(truth)
    if not t:
        return 0
    # correct if the ground-truth answer appears as a token/substring of the cleaned prediction
    if p == t:
        return 1
    # word-boundary containment (handles "the answer is 100000" and "100000.")
    if re.search(r'(?<!\w)' + re.escape(t) + r'(?!\w)', p):
        return 1
    return 0


# Synthetic fact generators:
COLORS = ["blue", "green", "red", "yellow", "purple"]
COMPANIES = ["Google", "Microsoft", "Amazon", "Meta", "Apple"]

def numerical_fact(i):
    person = f"Person_{i}"
    value = str(100000 + i * 137)
    return {
        "fact_type": "numerical",
        "text": f"{person}'s employee ID is {value}.",
        "question": f"What is {person}'s employee ID?",
        "answer": value,
    }

def entity_fact(i):
    person = f"Person_{i}"
    color = COLORS[i % len(COLORS)]
    return {
        "fact_type": "entity-attribute",
        "text": f"{person}'s favorite color is {color}.",
        "question": f"What is {person}'s favorite color?",
        "answer": color,
    }

def temporal_fact(i):
    a = f"Person_{i}"
    b = f"Person_{i+1}"
    return {
        "fact_type": "temporal",
        "text": f"{a} arrived before {b}.",
        "question": f"Who arrived first, {a} or {b}?",
        "answer": a,
    }

def multi_hop_fact(i):
    a = f"Person_{i}"
    b = f"Person_{i+1}"
    company = COMPANIES[i % len(COMPANIES)]
    return {
        "fact_type": "multi-hop",
        "text": (
            f"{a} manages {b}. "
            f"{b} works for {company}."
        ),
        "question": (
            f"Which company does {a}'s subordinate work for?"
        ),
        "answer": company,
    }

def contradictory_fact(i):
    person = f"Person_{i}"
    location1 = ["Paris", "Berlin", "Tokyo", "Delhi", "Sydney"][i % 5]
    location2 = ["London", "Madrid", "Beijing", "Toronto", "Dubai"][i % 5]
    return {
        "fact_type": "contradictory",
        "text": f"{person} lived in {location1}. {person} now lives in {location2}.",
        "question": f"Where does {person} live?",
        "answer": location2,
    }

# Map fact_type key to generator function:
FACT_GENERATORS = {
    "numerical": numerical_fact,
    "temporal": temporal_fact,
    "entity-attribute": entity_fact,
    "multi-hop": multi_hop_fact,
    "contradictory": contradictory_fact,
}

def build_sequence(fact_type, distractor_density, importance, seed, facts_after_target):
    """
    Generate a single target-first context sequence with repeated probes.
    Returns a list of probes; each probe is a dict as defined below.
    """
    random.seed(seed)
    np.random.seed(seed)
    # Generate the target fact (always index 0)
    target = FACT_GENERATORS[fact_type](0)
    target_fact = target.copy()
    target_fact["importance"] = importance
    target_fact["distractor_density"] = distractor_density
    target_fact["fact_id"] = f"{fact_type}_0_seed{seed}"

    # Build distractors list
    # Number of distractors is the maximum needed (max facts_after_target)
    max_facts = max(facts_after_target)
    distractors = []
    for i in range(1, max_facts + 1):
        # Choose fact type for distractor
        if distractor_density == "low":
            # random choice of any type except contradictory for simplicity
            dtype = random.choice(list(FACT_GENERATORS.keys()))
        else:
            # high density: concentrate on target type
            if random.random() < 0.8:
                dtype = fact_type
            else:
                dtype = random.choice(list(FACT_GENERATORS.keys()))
        distractor = FACT_GENERATORS[dtype](i)
        distractor["importance"] = importance  # propagate importance (not used by model)
        distractor["distractor_density"] = distractor_density
        distractors.append(distractor)

    # Now create probes at each required facts_after_target
    probes = []
    for probe_idx, aft in enumerate(sorted(facts_after_target)):
        # Compose current facts: target + first `aft` distractors
        current_facts = [target_fact] + distractors[:aft]
        # Build prompt text
        context = "\n".join(f"- {f['text']}" for f in current_facts)
        prompt = f"""You are given a set of factual statements.

{context}

Question:
{target_fact['question']}

Answer with only the short answer.
"""
        probes.append({
            "fact_id": target_fact["fact_id"],
            "fact_type": fact_type,
            "importance": importance,
            "distractor_density": distractor_density,
            "facts_after_target": aft,
            "prompt": prompt,
            "ground_truth": target_fact["answer"],
            "probe_index": probe_idx
        })
    return probes

def probe_target(probe):
    """
    Run the model on a single probe (prompt) and return prediction, correctness, and confidence.
    """
    prompt = probe["prompt"]
    inputs = tokenizer(prompt, return_tensors="pt", truncation=False).to(device)
    input_len = inputs["input_ids"].shape[1]
    # Generate one answer with greedy decoding
    outputs = model.generate(**inputs, max_new_tokens=12, do_sample=False,
                              return_dict_in_generate=True, output_scores=True,
                              pad_token_id=tokenizer.eos_token_id)
    # Extract the generated tokens and string
    gen_tokens = outputs.sequences[0, input_len:]
    pred = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
    # Compute log-probabilities of generated tokens
    transition_scores = model.compute_transition_scores(
        outputs.sequences, outputs.scores, normalize_logits=True
    )
    # Confidence: product of token probabilities (exp of sum of log-probs)
    log_probs = transition_scores[0][:len(gen_tokens)]
    logp_sum = log_probs.sum().item()
    confidence = float(np.exp(logp_sum))
    correct = answer_matches(pred, probe["ground_truth"])
    return pred, correct, confidence

def run_pilot(facts_after_steps=[0,25,50,100,200], trials_per_type=5):
    """
    Run the full pilot over all types, distractor densities, importances, and trials.
    Returns a DataFrame of results.
    """
    results = []
    total_runs = len(FACT_GENERATORS) * 2 * 2 * trials_per_type
    run_count = 0
    for fact_type in FACT_GENERATORS:
        for density in ["low", "high"]:
            for importance in ["low", "high"]:
                for trial in range(trials_per_type):
                    seed = trial + 1000*run_count
                    probes = build_sequence(fact_type, density, importance, seed, facts_after_steps)
                    for probe in probes:
                        run_count += 0 if probe["facts_after_target"] != facts_after_steps[0] else 1
                        prompt = probe["prompt"]
                        token_count = len(tokenizer.encode(prompt))
                        pred, corr, conf = probe_target(probe)
                        results.append({
                            "fact_id": probe["fact_id"],
                            "fact_type": probe["fact_type"],
                            "importance": probe["importance"],
                            "distractor_density": probe["distractor_density"],
                            "facts_after_target": probe["facts_after_target"],
                            "memory_condition": (
                                "in_window" if probe["facts_after_target"] == 0 else
                                "recently_compressed" if probe["facts_after_target"] <= 50 else
                                "heavily_compressed"
                            ),
                            "probe_index": probe["probe_index"],
                            "token_count": token_count,
                            "question": probe["prompt"].split("Question:")[1].strip(),
                            "ground_truth": probe["ground_truth"],
                            "prediction": pred,
                            "correct": corr,
                            "confidence": conf,
                            "seed": seed,
                            "prompt": probe["prompt"]
                        })
    df = pd.DataFrame(results)

    Path("ahn_pilot_results").mkdir(exist_ok=True)
    df.to_csv("ahn_pilot_results/raw_results.csv", index=False)
    return df


df_results = run_pilot()
print(df_results.head())


/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


             fact_id  fact_type importance distractor_density  \
0  numerical_0_seed0  numerical        low                low   
1  numerical_0_seed0  numerical        low                low   
2  numerical_0_seed0  numerical        low                low   
3  numerical_0_seed0  numerical        low                low   
4  numerical_0_seed0  numerical        low                low   

   facts_after_target     memory_condition  probe_index  token_count  \
0                   0            in_window            0           43   
1                  25  recently_compressed            1          410   
2                  50  recently_compressed            2          807   
3                 100   heavily_compressed            3         1584   
4                 200   heavily_compressed            4         3296   

                                            question ground_truth  \
0  What is Person_0's employee ID?\n\nAnswer with...       100000   
1  What is Person_0's employee ID?\n\n

In [12]:

PROMPT_EXAMPLE = """
The following facts are known:

- Alice's employee ID is 12345.
- Bob's favorite color is green.

Question:
What is Alice's employee ID?

Answer with only the short answer.
""".strip()

inputs = tokenizer(PROMPT_EXAMPLE, return_tensors="pt").to(device)
out = model.generate(**inputs, max_new_tokens=5, do_sample=False, return_dict_in_generate=True, output_scores=True)
transition_scores = model.compute_transition_scores(out.sequences, out.scores, normalize_logits=True)
predicted_tokens = out.sequences[0, inputs["input_ids"].shape[1]:]
pred_answer = tokenizer.decode(predicted_tokens, skip_special_tokens=True).strip()
# full-sequence confidence (mean-token prob), consistent with probe_target
gen_len = predicted_tokens.shape[0]
seq_logp = transition_scores[0][:gen_len].sum().item()
prob = float(np.exp(seq_logp))
print("Prompt:\n", PROMPT_EXAMPLE)
print("Predicted Answer:", pred_answer, "| Confidence:", prob)


/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


Prompt:
 The following facts are known:

- Alice's employee ID is 12345.
- Bob's favorite color is green.

Question:
What is Alice's employee ID?

Answer with only the short answer.
Predicted Answer: 1234 | Confidence: 0.9999990463265931


## Metrics (patched)\nRecall curves (H1/H2), **ECE + Brier** with a reliability diagram (H3), a confidently-wrong rate, and a per-fact-type calibration table. Answer matching is now normalized (punctuation/junk-tolerant), and confidence is the full-sequence token probability.

In [ ]:
# ===== Metrics: recall curves, ECE, Brier =====
import numpy as np
import matplotlib.pyplot as plt

# ---------- H1: per-fact-type recall vs compression ----------
summary = (df_results
           .groupby(["fact_type", "facts_after_target"])["correct"]
           .agg(["mean", "count"]).reset_index())

plt.figure(figsize=(7,4))
for fact in FACT_GENERATORS.keys():
    d = summary[summary.fact_type == fact]
    if len(d):
        plt.plot(d.facts_after_target, d["mean"], marker="o", label=fact)
plt.xlabel("Facts after target (compression pressure)")
plt.ylabel("Recall (accuracy)")
plt.title("H1/H2: recall vs compression, by fact type")
plt.ylim(-0.05, 1.05); plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig("ahn_pilot_results/recall_curves.png", dpi=120); plt.show()

# ---------- H3: calibration metrics ----------
def expected_calibration_error(confidences, correct, n_bins=10):
    """Guo et al. 2017 ECE with equal-width bins."""
    confidences = np.asarray(confidences, dtype=float)
    correct = np.asarray(correct, dtype=float)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    N = len(confidences)
    rows = []
    for b in range(n_bins):
        lo, hi = bins[b], bins[b+1]
        mask = (confidences > lo) & (confidences <= hi) if b > 0 else (confidences >= lo) & (confidences <= hi)
        if mask.sum() == 0:
            continue
        acc = correct[mask].mean()
        conf = confidences[mask].mean()
        w = mask.sum() / N
        ece += w * abs(acc - conf)
        rows.append((f"({lo:.1f},{hi:.1f}]", mask.sum(), acc, conf))
    return ece, rows

def brier_score(confidences, correct):
    """Mean squared error between confidence and 0/1 correctness."""
    confidences = np.asarray(confidences, dtype=float)
    correct = np.asarray(correct, dtype=float)
    return float(np.mean((confidences - correct) ** 2))

conf = df_results["confidence"].values
corr = df_results["correct"].values

ece, bin_rows = expected_calibration_error(conf, corr, n_bins=10)
brier = brier_score(conf, corr)

print(f"Overall ECE   : {ece:.4f}")
print(f"Overall Brier : {brier:.4f}")
print("\nReliability table (confidence bin | n | accuracy | mean confidence):")
for label, n, acc, c in bin_rows:
    flag = "  <-- overconfident" if c - acc > 0.1 else ""
    print(f"  {label:>10}  n={n:<4d}  acc={acc:.2f}  conf={c:.2f}{flag}")

# ---------- H3: reliability diagram ----------
plt.figure(figsize=(5,5))
if bin_rows:
    xs = [r[3] for r in bin_rows]  # mean confidence
    ys = [r[2] for r in bin_rows]  # accuracy
    plt.plot([0,1],[0,1],"--",color="gray",label="perfect calibration")
    plt.plot(xs, ys, marker="o", label="AHN")
plt.xlabel("Mean confidence"); plt.ylabel("Accuracy")
plt.title(f"H3: reliability diagram (ECE={ece:.3f}, Brier={brier:.3f})")
plt.xlim(0,1); plt.ylim(0,1); plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig("ahn_pilot_results/reliability_diagram.png", dpi=120); plt.show()

# ---------- confidently-wrong rate (H3, most direct) ----------
tau = 0.5
high_conf = df_results[df_results["confidence"] >= tau]
if len(high_conf):
    cw_rate = 1.0 - high_conf["correct"].mean()
    print(f"\nConfidently-wrong rate (conf>={tau}): {cw_rate:.3f} "
          f"({int((1-high_conf['correct']).sum())}/{len(high_conf)})")

# ---------- ECE/Brier broken down by fact type (feeds Table for H1+H3) ----------
print("\nPer-fact-type calibration:")
per_type = []
for ft in FACT_GENERATORS.keys():
    sub = df_results[df_results.fact_type == ft]
    if len(sub) == 0:
        continue
    e, _ = expected_calibration_error(sub["confidence"].values, sub["correct"].values, n_bins=10)
    b = brier_score(sub["confidence"].values, sub["correct"].values)
    per_type.append({"fact_type": ft, "n": len(sub),
                     "accuracy": round(sub["correct"].mean(),3),
                     "ECE": round(e,3), "Brier": round(b,3)})
per_type_df = pd.DataFrame(per_type)
print(per_type_df.to_string(index=False))
per_type_df.to_csv("ahn_pilot_results/per_type_calibration.csv", index=False)
